In [ ]:
print("hi this is tic tac toe")

In [ ]:
from enum import Enum

class CellState(Enum):
    EMPTY = ' '
    X = 'X'
    O = 'O'
    
class currentPlayer(Enum):
    PLAYER_X = CellState.X
    PLAYER_O = CellState.O
    
class status(Enum):
    IN_PROGRESS = 0
    PLAYER_X_WINS = 1
    PLAYER_O_WINS = 2
    DRAW = 3


In [ ]:
class rule_engine:
    def __init__(self):
        self.board = [[CellState.EMPTY for _ in range(3)] for _ in range(3)]
        self.current_player = currentPlayer.PLAYER_X
        self.status = status.IN_PROGRESS
        
    def get_current_player(self):
        return self.current_player
    
    def get_status(self):
        return self.status
    
    def update_status(self):
        # Check rows, columns, and diagonals for a win
        for i in range(3):
            if self.board[i][0] == self.board[i][1] == self.board[i][2] != CellState.EMPTY:
                self.status = status.PLAYER_X_WINS if self.board[i][0] == CellState.X else status.PLAYER_O_WINS
                return
            if self.board[0][i] == self.board[1][i] == self.board[2][i] != CellState.EMPTY:
                self.status = status.PLAYER_X_WINS if self.board[0][i] == CellState.X else status.PLAYER_O_WINS
                return
        if self.board[0][0] == self.board[1][1] == self.board[2][2] != CellState.EMPTY:
            self.status = status.PLAYER_X_WINS if self.board[0][0] == CellState.X else status.PLAYER_O_WINS
            return
        if self.board[0][2] == self.board[1][1] == self.board[2][0] != CellState.EMPTY:
            self.status = status.PLAYER_X_WINS if self.board[0][2] == CellState.X else status.PLAYER_O_WINS
            return
        
        # Check for draw
        if all(cell != CellState.EMPTY for row in self.board for cell in row):
            self.status = status.DRAW
            
    def get_legal_actions(self):
        legal_actions = []
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == CellState.EMPTY:
                    legal_actions.append((i, j))
        return legal_actions
    
    def apply_action(self, action):
        if self.status != status.IN_PROGRESS:
            raise Exception("Game is already over.")
        if action not in self.get_legal_actions():
            raise Exception("Invalid action.")
        i, j = action
        self.board[i][j] = self.current_player.value
        self.update_status()
        self.current_player = currentPlayer.PLAYER_O if self.current_player == currentPlayer.PLAYER_X else currentPlayer.PLAYER_X
    
    def is_game_over(self):
        return self.status != status.IN_PROGRESS

In [ ]:
def print_board(board):
    for row in board:
        print(' | '.join(cell.value for cell in row))
        print('-' * 5)

In [ ]:
import copy

def human_agent(engine) -> tuple:
    legal_actions = engine.get_legal_actions()
    if not legal_actions:
        return None
    print("Legal actions:", legal_actions)
    print_board(engine.board)
    while True:
        try:
            action = input("Enter your move as 'row,col': ")
            i, j = map(int, action.split(','))
            if (i, j) in legal_actions:
                return (i, j)
            else:
                print("Invalid move. Try again.")
        except Exception as e:
            print(f"Error: {e}. Please enter a valid move.")

def random_agent(engine) -> tuple:
    import random
    legal_actions = engine.get_legal_actions()
    if legal_actions:
        return random.choice(legal_actions)
    return None

def random_wrapper(engine, decision_metrics):
    return random_agent(engine)



In [ ]:
def minimax_search(
    engine,
    maximizing_player,
    search_metrics,
    depth=0
):
    # We entered one search node
    search_metrics.nodes_explored += 1
    # Track deepest level reached
    search_metrics.max_depth = max(
        search_metrics.max_depth,
        depth
    )
    # Terminal state
    if engine.is_game_over():
        search_metrics.terminal_nodes += 1
        if engine.get_status() == status.PLAYER_X_WINS:
            return 1
        elif engine.get_status() == status.PLAYER_O_WINS:
            return -1
        else:
            return 0
    # Maximizing player
    if maximizing_player:
        max_eval = float('-inf')
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            search_metrics.deep_copies += 1
            new_engine.apply_action(action)
            eval_score = minimax_search(
                new_engine,
                False,
                search_metrics,
                depth + 1
            )
            max_eval = max(max_eval, eval_score)
        return max_eval
    # Minimizing player
    else:
        min_eval = float('inf')
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            search_metrics.deep_copies += 1
            new_engine.apply_action(action)
            eval_score = minimax_search(
                new_engine,
                True,
                search_metrics,
                depth + 1
            )
            min_eval = min(min_eval, eval_score)
        return min_eval
    
def minmax_agent(engine, maximizing_player, search_metrics):
    best_action = None
    if maximizing_player:
        best_eval = float('-inf')
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            search_metrics.deep_copies += 1
            new_engine.apply_action(action)
            eval_score = minimax_search(
                new_engine,
                False,
                search_metrics,
                depth=1
            )
            if eval_score > best_eval:
                best_eval = eval_score
                best_action = action
    else:
        best_eval = float('inf')
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            search_metrics.deep_copies += 1
            new_engine.apply_action(action)
            eval_score = minimax_search(
                new_engine,
                True,
                search_metrics,
                depth=1
            )
            if eval_score < best_eval:
                best_eval = eval_score
                best_action = action
    return best_action

def minimax_wrapper(engine, decision_metrics, maximizing_player):
    return minmax_agent(
        engine,
        maximizing_player,
        decision_metrics.search_metrics
    )

def minimax_x_wrapper(engine, decision_metrics):
    return minmax_agent(
        engine,
        True,
        decision_metrics.search_metrics
    )

def minimax_o_wrapper(engine, decision_metrics):
    return minmax_agent(
        engine,
        False,
        decision_metrics.search_metrics
    )

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SearchMetrics:
    nodes_explored: int = 0
    terminal_nodes: int = 0
    deep_copies: int = 0
    max_depth: int = 0

@dataclass
class DecisionMetrics:
    player: str
    agent: str
    duration_ms: float = 0.0
    search_metrics: SearchMetrics = field(default_factory=SearchMetrics)
    chosen_action: Optional[tuple] = None

@dataclass
class MatchMetrics:
    player_x_agent: str
    player_o_agent: str
    winner: status = status.IN_PROGRESS
    no_of_moves: int = 0
    duration_ms: float = 0.0
    decisions: list[DecisionMetrics] = field(default_factory=list)
    
def print_match_metrics(metrics: MatchMetrics):

    print("\n" + "=" * 70)
    print("MATCH REPORT")
    print("=" * 70)

    print(f"PLAYER X     : {metrics.player_x_agent}")
    print(f"PLAYER O     : {metrics.player_o_agent}")
    print(f"WINNER       : {metrics.winner.name}")
    print(f"MOVES        : {metrics.no_of_moves}")
    print(f"DURATION     : {metrics.duration_ms:.2f} ms")
    print(f"DURATION     : {metrics.duration_ms / 1000:.3f} s")

    print("\n" + "-" * 70)
    print("DECISIONS")
    print("-" * 70)

    for i, decision in enumerate(metrics.decisions, start=1):

        search = decision.search_metrics

        print(f"\nDecision {i}")
        print(f"  Player          : {decision.player}")
        print(f"  Agent           : {decision.agent}")
        print(f"  Action          : {decision.chosen_action}")
        print(f"  Duration        : {decision.duration_ms:.3f} ms")

        print("  Search:")
        print(f"    Nodes explored : {search.nodes_explored:,}")
        print(f"    Terminal nodes : {search.terminal_nodes:,}")
        print(f"    Deep copies    : {search.deep_copies:,}")
        print(f"    Max depth      : {search.max_depth}")

    print("\n" + "=" * 70)

In [ ]:
import time

def matchup(
    engine,
    agent1,
    agent2,
    agent1_name="Agent 1",
    agent2_name="Agent 2"
) -> MatchMetrics:
    match_metrics = MatchMetrics(
        player_x_agent=agent1_name,
        player_o_agent=agent2_name
    )
    match_start = time.perf_counter()
    while not engine.is_game_over():
        current_player = engine.get_current_player()
        if current_player == currentPlayer.PLAYER_X:
            agent = agent1
            agent_name = agent1_name
            player_name = "PLAYER_X"
        else:
            agent = agent2
            agent_name = agent2_name
            player_name = "PLAYER_O"
            
        decision_metrics = DecisionMetrics(
            player=player_name,
            agent=agent_name
        )
        # -------------------------
        # Execute agent
        # -------------------------
        decision_start = time.perf_counter()
        action = agent(engine, decision_metrics)
        decision_end = time.perf_counter()
        # -------------------------
        # Finish decision metrics
        # -------------------------
        decision_metrics.duration_ms = (
            decision_end - decision_start
        ) * 1000
        decision_metrics.chosen_action = action
        match_metrics.decisions.append(
            decision_metrics
        )
        # -------------------------
        # Apply action
        # -------------------------
        if action is not None:
            engine.apply_action(action)
            match_metrics.no_of_moves += 1
        else:
            break
    # -------------------------
    # Finish match metrics
    # -------------------------
    match_end = time.perf_counter()
    match_metrics.duration_ms = (
        match_end - match_start
    ) * 1000
    match_metrics.winner = engine.get_status()
    return match_metrics

In [ ]:
metrics = matchup(
    rule_engine(),
    minimax_x_wrapper,
    random_wrapper,
    "Minimax",
    "Random"
)

print_match_metrics(metrics)

In [ ]:
# game loop , minmax agent vs random agent
x_count = 0
o_count = 0
draw_count = 0

for _ in range(1):  # Play 1 game
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            _,action = minmax_agent(engine, True)
        else:
            action = random_agent(engine)
        if action:
            engine.apply_action(action)

    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        o_count += 1
    else:
        draw_count += 1


print(f"Player X wins(minmax): {x_count}")
print(f"Player O wins(random): {o_count}")
print(f"Draws: {draw_count}")

In [ ]:
x_count=0
y_count=0
draw_count=0

for _ in range(5):  # Play 5 games
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            action = random_agent(engine)
        else:
            _,action = minmax_agent(engine, False)
        if action:
            engine.apply_action(action)

    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        y_count += 1
    else:
        draw_count += 1

print(f"Player X wins(random): {x_count}")
print(f"Player O wins(minmax): {y_count}")
print(f"Draws: {draw_count}")

In [ ]:
x_count=0
y_count=0
draw_count=0

for _ in range(1):  # Play 1 game
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            _,action = minmax_agent(engine, True)
        else:
            _,action = minmax_agent(engine, False)
        if action:
            engine.apply_action(action)

    print(f"Result: {engine.get_status()}")
    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        y_count += 1
    else:
        draw_count += 1

print(f"Player X wins(minmax): {x_count}")
print(f"Player O wins(minmax): {y_count}")
print(f"Draws: {draw_count}")

In [ ]:
x_count=0
y_count=0
draw_count=0

engine = rule_engine()
while not engine.is_game_over():
    if engine.get_current_player() == currentPlayer.PLAYER_X:
        action = human_agent(engine)
    else:
        _,action = minmax_agent(engine, False)
    if action:
        engine.apply_action(action)
    